In [2]:
import json
import re
# import torch
# import torchaudio
import soundfile as sf
from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

/Users/edwardyang/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def clean_text(text):
    text = text.replace("\n", " ")
    # # Remove speaker names (e.g., "RACHEL:" or "ROSS & PHOEBE:")
    # text = re.sub(r"^[A-Z &]+ ?: ?", "", text, flags=re.MULTILINE)
    # Remove speaker names (e.g., "RACHEL:" or "ROSS & PHOEBE:")
    text = re.sub(r"[A-Z &]+: ?", " ", text)

    # Expand contractions
    contractions = {
        "it's": "it is", "you're": "you are", "i'm": "i am", "don't": "do not",
        "won't": "will not", "she's": "she is", "he's": "he is", "we're": "we are",
        "they're": "they are", "can't": "cannot", "didn't": "did not", "i've": "i have",
        "wasn't": "was not", "isn't": "is not", "aren't": "are not", "let's": "let us",
        "what's": "what is", "there's": "there is", "that's": "that is", "it'd": "it would",
        "you'd": "you would", "i'd": "i would"
    }
    for contraction, expanded in contractions.items():
        text = re.sub(rf"\b{re.escape(contraction)}\b", expanded, text, flags=re.IGNORECASE)

    # Remove all punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Convert to uppercase
    text = text.upper()

    return text


In [4]:
# This is a cleaned json file for audio clips in part_1
data_split = './Data/violin_audios/data_subtitles_TV.json'
audio_dir = "./Data/violin_audios/audio_clips_16K/"


# This is a cleaned json file for audio clips in part_1, sperate into segments defined by subtitles
# part_1_short_split = '/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/data_subtitles_split_1_short_TV.json'
# audio_dir = "/content/drive/MyDrive/EE6366_Speech_Speaker_Recognition/VIOLIN_Audio_Data/audio_clips_part_1_short/"

with open(data_split) as f:
  data = json.load(f)

In [5]:
# Define the mapping function
def read_sound(batch):
    speech, sr = sf.read(batch["file"])
    assert sr == 16000
    batch["audio"] = {
        "array": speech,  # Convert to NumPy array for serialization
        "sampling_rate": 16000,
    }
    # print(batch)
    return batch

In [6]:
split_data = {
    "train": [],
    "validate": [],
    "test": []
}

for key in tqdm(data,desc="Preprocessing dataset"):
  # try:
  data_path = audio_dir + key + ".flac"
  # array = downsample_audio(data_path)
  split = data[key]['split']  # Get the split (train/validation/test)
  # print(split)
  # Append data to the corresponding split
  split_data[split].append(
    {
      "file": data_path,
      "name": key,
      # "audio": {
      #   "array": array,
      #   "sampling_rate": 16000,
      # },
      "text": clean_text(data[key]['sub']),
      # "duration": data[key]['duration'],
      # "speech_duration": data[key]['speech_duration'],
    }
  )
  # except:
  #   continue


# Create DatasetDict from the split_data
dataset = DatasetDict(
    {split: Dataset.from_list(data_list) for split, data_list in split_data.items()}
)

Preprocessing dataset: 100%|██████████| 10003/10003 [00:01<00:00, 6514.85it/s]


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['file', 'name', 'text'],
        num_rows: 7983
    })
    validate: Dataset({
        features: ['file', 'name', 'text'],
        num_rows: 1007
    })
    test: Dataset({
        features: ['file', 'name', 'text'],
        num_rows: 1013
    })
})

In [8]:
dataset = dataset.map(read_sound, num_proc=1)

Map: 100%|██████████| 1013/1013 [00:13<00:00, 72.58 examples/s]


In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['file', 'name', 'text', 'audio'],
        num_rows: 7983
    })
    validate: Dataset({
        features: ['file', 'name', 'text', 'audio'],
        num_rows: 1007
    })
    test: Dataset({
        features: ['file', 'name', 'text', 'audio'],
        num_rows: 1013
    })
})

In [11]:
dataset.save_to_disk("Data/violin_audios/dataset_clean")

Saving the dataset (44/67 shards):  66%|██████▌   | 5246/7983 [02:41<01:24, 32.43 examples/s]


KeyboardInterrupt: 

In [33]:
sf.read('./Data/violin_audios/audio_clips_16K/friends_s05e21_clip_686_722.flac')

(array([ 0.00140381,  0.00097656, -0.00012207, ...,  0.00421143,
         0.00244141, -0.00128174]),
 16000)

In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 7983
    })
    validate: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 1007
    })
    test: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 1013
    })
})

In [23]:
# Define the sampling fraction
sampling_fraction = 0.1

# Randomly sample 10% of each split
sampled_dataset = DatasetDict({
    "train": dataset["train"].train_test_split(test_size=sampling_fraction)["test"],
    "validate": dataset["validate"].train_test_split(test_size=sampling_fraction)["test"],
    "test": dataset["test"].train_test_split(test_size=sampling_fraction)["test"],
})

# Inspect the sampled dataset
print(sampled_dataset)

DatasetDict({
    train: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 799
    })
    validate: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 101
    })
    test: Dataset({
        features: ['file', 'name', 'text', 'duration', 'speech_duration', 'audio'],
        num_rows: 102
    })
})


In [25]:
sampled_dataset.save_to_disk("Data/violin_audios/dataset_small")

Saving the dataset (1/1 shards): 100%|██████████| 102/102 [00:00<00:00, 235.87 examples/s]
